# 蝶威量化因子挖掘大赛 - 最优方案

---

## 一、比赛内容是什么？我该完成什么？

**用通俗的话来说：**

这是一个由上海蝶威私募基金（DeepWin）举办的量化投资比赛。你的任务就像是当一个「股票预测员」——

主办方给你A股沪深300成分股在2023-2024年间的**高频交易数据**（每隔几秒就记录一次的股价、买卖挂单量等信息），你需要从中找到一个「规律」或「信号」（称为**因子**），来预测哪些股票接下来会涨。

**具体你需要做的：**
1. 利用平台提供的分钟级快照数据（包含价格、成交量、10档买卖盘口等）
2. 构建一个**15分钟频率的因子**——即每15分钟给每只股票打一个分数
3. 将代码写在 `main` 函数中，输出格式为 `[date, instrument, factor]` 三列
4. 因子值越大代表越看好该股票（系统会选因子值最大的60只股票买入）
5. 系统会自动做回测：用你的因子选股，看超额收益和夏普比率

**评分公式：**
- `Final Score = 0.3 × 单因子分析排名 + 0.7 × 单因子回测排名`
- 回测会在16个不同的截断时间点（从09:45到15:00）分别测试，取均值
- 因子会先被BARRA风险因子中性化（取残差），所以你的因子需要有独立的增量信息

**限制条件：**
- CPU Notebook运行时间 ≤ 9小时
- 禁止外部网络和未授权数据
- 提交的是可运行代码，不是数据文件

---

## 二、官方Demo的逻辑是什么？

官方Demo（`demo/demo.ipynb`）构建的是一个**加权订单簿压力因子**，逻辑如下：

1. **计算中间价**：`mid_price = (ask_price1 + bid_price1) / 2`

2. **5档指数加权挂单量**：
   - 买方：`W_bid = bid_vol1 × 1.0 + bid_vol2 × e^(-0.3) + ... + bid_vol5 × e^(-1.2)`
   - 卖方：同理计算 `W_ask`
   - 越靠近盘口（一档）的挂单权重越高

3. **订单簿不平衡度**：`imbalance = (W_bid - W_ask) / (W_bid + W_ask)`

4. **相对价差**：`spread = (ask1 - bid1) / mid_price`

5. **原始压力因子**：`raw_pressure = imbalance² / sqrt(|spread|)`
   - 不平衡度越大、价差越小，压力越大

6. **15分钟窗口内处理**：
   - Z-Score标准化：`z = (last - mean) / std`
   - 波动率调整：高波动时因子值 × 0.7
   - tanh缩放到(-1, 1)

7. **因子方向取反** (`× -1`)：因为卖压大时预期下跌

**核心思想**：通过买卖盘口的力量对比来预测短期价格方向。

---

## 三、其余文件的策略分析

### 3.1 订单簿压力因子（`xyf/订单薄压力.ipynb`）
- **策略**：与Demo基本相同，但 `raw_pressure = imbalance / sqrt(|spread|)`（没有对imbalance求平方），且因子方向为正(×1)
- **优点**：直接利用微观结构信息，信号最直接
- **缺点**：在高波动市场中噪音大；容易受到大单冲击的影响；可能被BARRA因子吸收

### 3.2 动量反转因子（`xyf/动量反转.ipynb`）
- **策略**：计算15分钟窗口内收盘中间价相对均价的偏离度，取反（均值回归逻辑）
- **公式**：`factor = -tanh((close - avg) / avg × 10)`
- **优点**：经典日内均值回归逻辑，简洁有效；tanh处理避免极端值
- **缺点**：单一信号维度，缺乏微观结构信息；放大系数10是经验值，不一定最优

### 3.3 价格排序因子（`xyf/价格排序.ipynb`）
- **策略**：每个15分钟截面内按收盘中间价从高到低排序，低价股得分高
- **公式**：`factor = 1 - (N - rank_desc) / (N - 1)`，范围[0,1]
- **优点**：逻辑简单清晰，不依赖复杂计算
- **缺点**：低价效应可能被BARRA的市值因子完全中性化掉；纯截面排序缺乏时序信息

### 3.4 价格&反转复合因子（`xyf/价格&反转复合.ipynb`）
- **策略**：将价格排序因子(20%)和动量反转因子(80%)加权组合
- **优点**：多信号融合比单因子更稳健；反转因子占主导，价格因子作补充
- **缺点**：权重0.2/0.8是人工设定；两个子因子相关性可能较高；缺乏订单簿微观信息

### 3.5 量价相关性因子（`xyf/量价相关性.ipynb`）
- **策略**：计算15分钟窗口内价格收益率与成交量变化的Pearson相关系数
- **公式**：`factor = Corr(returns, volume_delta)`
- **优点**：捕捉量价关系这一正交维度；能反映资金流向与价格联动性
- **缺点**：窗口内样本量有限，相关系数估计不稳定；对极端值敏感

### 3.6 momentum_py-v3（`xyf/momentum_py-v3.ipynb`）
- **策略**：与动量反转因子相同逻辑，但使用Python分块回测引擎（含并行版本）
- **优点**：回测引擎优化好，适合快速验证
- **缺点**：因子逻辑本身没有改进

### 3.7 回测模块汇总（`xyf/回测模块汇总.ipynb`）
- **内容**：三种回测方式的代码模板（基础分块版、因子并行版、全并行版）
- **用途**：不包含因子逻辑，仅作为回测执行框架

---

## 四、最优方案：多维微观结构复合因子

### 设计思路

综合已有策略的优缺点，最优方案应当：

1. **融合多维信号**：订单簿压力 + 日内反转 + 量价相关性，三个正交维度
2. **充分利用10档盘口数据**：扩展到10档加权不平衡度，信息更充分
3. **自适应波动率调整**：根据窗口内波动率动态衰减因子强度
4. **截面标准化**：在每个时间截面上进行Z-Score标准化，消除尺度差异
5. **纯SQL实现**：使用DAI数据引擎直接计算，速度快、内存低

### 因子公式

**子因子1 - 10档加权订单簿不平衡度（Order Book Imbalance, OBI）：**

$$W_{bid} = \sum_{i=1}^{10} bid\_volume_i \times e^{-0.2(i-1)}$$
$$OBI = \frac{W_{bid} - W_{ask}}{W_{bid} + W_{ask}}$$

取15分钟窗口均值后做Z-Score标准化。

**子因子2 - 日内均值回归（Mean Reversion, MR）：**

$$MR = -\tanh\left(\frac{P_{close} - P_{avg}}{P_{avg}} \times 10\right)$$

**子因子3 - 量价相关性（PV Correlation, PVC）：**

$$PVC = \text{Corr}(r_t, \Delta V_t)$$

**复合因子：**

$$Factor = 0.45 \times OBI_{z} + 0.35 \times MR_{z} + 0.20 \times PVC_{z}$$

其中下标 $z$ 表示截面Z-Score标准化后的值。

最终通过 `tanh` 缩放并乘以波动率调整系数。

In [ ]:
def main(datasource, start_date, end_date):
    """
    多维微观结构复合因子 (Multi-Dimensional Microstructure Composite Factor)

    融合三个正交维度的信号：
    1. 10档加权订单簿不平衡度 (Order Book Imbalance)
    2. 日内均值回归动量 (Intraday Mean Reversion)
    3. 量价相关性 (Price-Volume Correlation)

    通过截面Z-Score标准化后加权组合，并经波动率自适应调整。

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # 子因子权重
    w_obi = 0.45   # 订单簿不平衡度权重
    w_mr  = 0.35   # 均值回归权重
    w_pvc = 0.20   # 量价相关性权重

    sql = f"""
    SET preserve_insertion_order=false;
    SET threads=4;

    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            volume,

            -- 中间价
            (ask_price1 + bid_price1) / 2.0 AS mid_price,

            -- 交易日
            strftime(date, '%Y-%m-%d') AS trading_day,

            -- 10档指数加权买方量 (衰减系数0.2)
            (
                COALESCE(bid_volume1, 0) * 1.0 +
                COALESCE(bid_volume2, 0) * EXP(-0.2) +
                COALESCE(bid_volume3, 0) * EXP(-0.4) +
                COALESCE(bid_volume4, 0) * EXP(-0.6) +
                COALESCE(bid_volume5, 0) * EXP(-0.8) +
                COALESCE(bid_volume6, 0) * EXP(-1.0) +
                COALESCE(bid_volume7, 0) * EXP(-1.2) +
                COALESCE(bid_volume8, 0) * EXP(-1.4) +
                COALESCE(bid_volume9, 0) * EXP(-1.6) +
                COALESCE(bid_volume10, 0) * EXP(-1.8)
            ) AS weight_bid,

            -- 10档指数加权卖方量
            (
                COALESCE(ask_volume1, 0) * 1.0 +
                COALESCE(ask_volume2, 0) * EXP(-0.2) +
                COALESCE(ask_volume3, 0) * EXP(-0.4) +
                COALESCE(ask_volume4, 0) * EXP(-0.6) +
                COALESCE(ask_volume5, 0) * EXP(-0.8) +
                COALESCE(ask_volume6, 0) * EXP(-1.0) +
                COALESCE(ask_volume7, 0) * EXP(-1.2) +
                COALESCE(ask_volume8, 0) * EXP(-1.4) +
                COALESCE(ask_volume9, 0) * EXP(-1.6) +
                COALESCE(ask_volume10, 0) * EXP(-1.8)
            ) AS weight_ask,

            -- 加权订单簿不平衡度
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) AS obi_raw,

            -- 相对价差
            (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2.0 + 1e-8) AS relative_spread,

            -- 15分钟时间窗口
            CASE
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END AS time_segment

        FROM {datasource}
        WHERE time_segment != -1
    ),

    -- 计算逐笔差分序列（用于量价相关性和波动率）
    cte_delta AS (
        SELECT
            *,
            LAG(mid_price) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS prev_mid_price,
            LAG(volume) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS prev_volume,

            -- 收益率
            (mid_price - prev_mid_price) / (prev_mid_price + 1e-8) AS ret,
            -- 成交量变化
            (volume - prev_volume) AS vol_delta,
            -- 对数收益率（用于波动率计算）
            ABS(mid_price / (prev_mid_price + 1e-8) - 1) AS abs_ret
        FROM cte_snapshot
    ),

    -- 15分钟窗口聚合：计算三个子因子的原始值
    cte_window AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,

            -- 子因子1: 订单簿不平衡度（窗口均值）
            AVG(obi_raw) AS obi_mean,

            -- 子因子2: 均值回归动量
            argMax(mid_price, date) AS close_mid,
            AVG(mid_price) AS avg_mid,
            -1.0 * tanh(((close_mid - avg_mid) / (avg_mid + 1e-8)) * 10.0) AS mr_raw,

            -- 子因子3: 量价相关性
            CASE
                WHEN COUNT(CASE WHEN prev_mid_price IS NOT NULL AND prev_volume IS NOT NULL THEN 1 END) < 3
                THEN 0
                ELSE COALESCE(CORR(ret, vol_delta), 0)
            END AS pvc_raw,

            -- 波动率（用于自适应调整）
            nanstd(ret) AS volatility,
            CASE
                WHEN COUNT(*) > 10 THEN quantile(abs_ret, 0.8)
                ELSE 0.01
            END AS vol_threshold

        FROM cte_delta
        GROUP BY instrument_id, trading_day, time_segment
    ),

    -- 截面Z-Score标准化
    cte_zscore AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,
            volatility,
            vol_threshold,

            -- OBI的截面Z-Score
            (obi_mean - AVG(obi_mean) OVER w) / (nanstd(obi_mean) OVER w + 1e-8) AS obi_z,

            -- MR的截面Z-Score
            (mr_raw - AVG(mr_raw) OVER w) / (nanstd(mr_raw) OVER w + 1e-8) AS mr_z,

            -- PVC的截面Z-Score
            (pvc_raw - AVG(pvc_raw) OVER w) / (nanstd(pvc_raw) OVER w + 1e-8) AS pvc_z

        FROM cte_window
        WINDOW w AS (PARTITION BY trading_day, time_segment)
    ),

    -- 复合因子计算
    cte_composite AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,

            -- 加权组合
            ({w_obi} * obi_z + {w_mr} * mr_z + {w_pvc} * pvc_z) AS raw_composite,

            -- 波动率自适应调整
            CASE
                WHEN volatility > vol_threshold AND vol_threshold > 0
                THEN raw_composite * 0.7
                ELSE raw_composite
            END AS adjusted_composite,

            -- tanh缩放到(-1, 1)
            tanh(adjusted_composite) AS factor

        FROM cte_zscore
    )

    -- 最终输出
    SELECT
        CAST(CONCAT(
            f.trading_day, ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor
    FROM cte_composite f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}).df()
    return df


if __name__ == '__main__':
    """
    开发调试专用模块：分块并行回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc
    from concurrent.futures import ThreadPoolExecutor, as_completed

    logger = structlog.get_logger()
    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'

    full_start_date = '2023-01-01'
    full_end_date = '2024-12-01'

    # 按月分块
    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq='MS')
    chunk_params = []
    for start_dt in date_ranges:
        current_start = start_dt.strftime('%Y-%m-%d 00:00:00')
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d 23:59:59')
        chunk_params.append((current_start, current_end))

    all_results = []
    max_workers = 4

    logger.info(f"Starting Composite Factor Backtest: {full_start_date} to {full_end_date}")
    logger.info(f"Parallel Workers: {max_workers}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_date = {
            executor.submit(main, datasource, start, end): (start, end)
            for start, end in chunk_params
        }

        for future in as_completed(future_to_date):
            start, end = future_to_date[future]
            try:
                df_chunk = future.result()
                if df_chunk is not None and not df_chunk.empty:
                    all_results.append(df_chunk)
                    logger.info(f"Chunk Done: {start[:7]} | Rows: {len(df_chunk)}")
                else:
                    logger.warning(f"Chunk Empty: {start[:7]}")
                del df_chunk
                gc.collect()
            except Exception as e:
                logger.error(f"Error in chunk {start}: {e}")

    if all_results:
        logger.info("Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=['date', 'instrument'], inplace=True)

        logger.info(f"All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")

        logger.info("Starting Evaluation...")
        try:
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed: {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")